# ROSALIA on PadChest-GR — uncertainty-stratified IoU eval

End-to-end Colab notebook. For every (image, grounded finding) pair in
PadChest-GR whose finding maps to one of ROSALIA's 7 supported lesion
concepts, we:

1. map the finding label to one of {cardiomegaly, pneumonia, atelectasis,
   opacity, consolidation, edema, effusion} — the ROSALIA vocabulary,
2. build a ROSALIA-style instruction, e.g. "Segment the pneumonia.",
3. run ROSALIA (LISA-7B + SAM-H fine-tuned on 1.1M MIMIC-ILS pairs) to get
   a predicted lesion mask,
4. compute a bundle of IoUs against the ground-truth annotator boxes (per
   box, intersection, pixel-OR union, outer-bbox union, pairwise
   inter-annotator agreement, reader-vs-reader),
5. aggregate the bundle stratified by the rule-based spatial-uncertainty
   label from `rule_uncertainty_scores.jsonl`.

**Why ROSALIA and not MedSAM-3?**
See §5 of arXiv:2511.19046 (MedSAM-3) and §6 of arXiv:2511.15186 (ROSALIA).
Text-only MedSAM-3 on CXR scores ~0.03 Dice; its LoRA was trained on
non-CXR modalities. ROSALIA is purpose-built for CXR lesion segmentation
via natural-language instructions and reports gIoU=0.712 on MIMIC-ILS.

Inputs:
* `gs://${BUCKET}/images/{image_id}` — extracted PadChest-GR PNGs.
* `project/data/processed/samples_with_uncertainty_and_iou.csv` — sample
  list with reader1/reader2 boxes, finding label, MedGemma uncertainty
  label.
* `project/data/processed/rule_uncertainty_scores.jsonl` — rule-based
  uncertainty label (used for stratification).
* `checkone/ROSALIA-7B-v1` — HuggingFace weights (public, ungated).

Outputs (per sample, written to GCS):
* `gs://${BUCKET}/outputs/rosalia_v1/{sample_id}.json` — IoU bundle.
* `gs://${BUCKET}/outputs/rosalia_v1/masks/{sample_id}.npz` — predicted binary mask.

Outputs (aggregated, copied back to the repo via Drive):
* `project/outputs/rosalia_per_sample_ious.csv`
* `project/outputs/rosalia_aggregate_ious.json`
* `project/outputs/rosalia_per_finding_ious.csv`
* `project/figures/rosalia_iou_by_uncertainty_group.png`
* `project/figures/rosalia_iou_histogram_by_group.png`
* `project/figures/rosalia_agreement_vs_iou_scatter.png`


## Cell 0 — one-time GCS extraction of the split-zip PadChest-GR archive

The dataset was uploaded as `gs://${BUCKET}/Padchest_GR_files/PadChest_GR.zip.001..037`.
This cell is idempotent: if `gs://${BUCKET}/images/` is already populated, it no-ops.

Uses modern `gcloud storage` CLI (not legacy `gsutil`) because it respects
Application Default Credentials seeded by `auth.authenticate_user()`.


In [ ]:
BUCKET      = "yang-padchest-gr"
GCS_PROJECT = "yang-uncertainty-eval"
SRC_PREFIX  = "Padchest_GR_files"
DST_PREFIX  = "images"

from google.colab import auth
auth.authenticate_user()
get_ipython().system(f"gcloud config set project {GCS_PROJECT} -q")
print("Active identity:")
get_ipython().system("gcloud auth list --filter=status:ACTIVE --format='value(account)'")

import subprocess
r = subprocess.run(
    f"gcloud storage ls gs://{BUCKET}/ 2>&1 | head -10",
    capture_output=True, text=True, shell=True,
)
print(f"\nPreflight `gcloud storage ls gs://{BUCKET}/`:")
print(r.stdout)
if "ERROR" in r.stdout or "403" in r.stdout or "404" in r.stdout or "401" in r.stdout:
    raise RuntimeError(
        f"Cannot read gs://{BUCKET}/ as the active account. Verify:\n"
        f"  - bucket name '{BUCKET}' is exactly right\n"
        f"  - the auth popup signed in as the account that owns the bucket\n"
        f"  - the bucket is in project '{GCS_PROJECT}'\n"
    )

from google.cloud import storage
_client = storage.Client(project=GCS_PROJECT)
_bucket = _client.bucket(BUCKET)

def gcs_has_images():
    blobs = list(_bucket.list_blobs(prefix=f"{DST_PREFIX}/", max_results=1))
    return len(blobs) > 0

if gcs_has_images():
    print(f"\ngs://{BUCKET}/{DST_PREFIX}/ already populated — skipping extraction.")
else:
    print("\nExtracting PadChest-GR from split-zip parts in the bucket. One-time slow step.")
    !mkdir -p /content/zips /content/images /content/extracted
    get_ipython().system(
        f"gcloud storage cp 'gs://{BUCKET}/{SRC_PREFIX}/PadChest_GR.zip.*' /content/zips/"
    )
    !ls /content/zips/PadChest_GR.zip.* | sort > /content/zip_parts.txt
    !cat $(cat /content/zip_parts.txt | tr '\n' ' ') > /content/full.zip
    !rm -f /content/zips/PadChest_GR.zip.*
    !unzip -q -o /content/full.zip -d /content/extracted/
    !rm -f /content/full.zip
    !find /content/extracted -type f -name '*.png' -print0 | xargs -0 -I{} cp {} /content/images/
    get_ipython().system(
        f"gcloud storage rsync /content/images/ gs://{BUCKET}/{DST_PREFIX}/ --recursive"
    )
    !rm -rf /content/extracted /content/images
    print("Extraction + upload complete.")


## Cell 1 — install ROSALIA dependencies (Colab)

ROSALIA needs a very specific dependency stack:
* `transformers==4.31.0` — this is the version LISA-7B was trained against.
  Newer transformers break the custom `LISAForCausalLM` class (attention
  API + tokenizer padding rules changed).
* `peft==0.4.0`, `einops==0.4.1`, `bitsandbytes==0.48.2` — same reason.
* `torch/torchvision/torchaudio` on CUDA 12.6 wheels.

We also clone the ROSALIA repo into `/content/rosalia_repo` so we can
`sys.path.insert` its custom `model/` and `utils/` packages in Cell 6.

**Runtime requirements**:
* GPU with ≥16 GB VRAM. ROSALIA-7B in bf16 needs ~15 GB just for weights
  (2 sharded .bin files: 9.98 GB + 6.14 GB). Add SAM-H decoder (~2 GB) and
  activations, and a Colab T4 (15 GB) is borderline OOM. **Use L4 (24 GB)
  or A100 (40 GB) for reliable inference**.
* After running this cell you MUST restart the runtime (pip cannot hot-swap
  transformers while a torch process holds it).


In [ ]:
# CUDA-12.6 PyTorch wheels (LISA/ROSALIA compatible)
!pip -q install --force-reinstall --no-deps \
    torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu126

# Clone ROSALIA repo (has custom model/, utils/ packages we import in Cell 6)
import os
if not os.path.isdir('/content/rosalia_repo'):
    !git clone https://github.com/checkoneee/ROSALIA.git /content/rosalia_repo

# ROSALIA's pinned dependencies. `transformers==4.31.0` is CRITICAL — the
# custom LISAForCausalLM class in model/LISA.py uses attention APIs that
# were removed in transformers >= 4.40.
!pip -q install \
    'transformers==4.31.0' \
    'peft==0.4.0' \
    'einops==0.4.1' \
    'sentencepiece' \
    'opencv-python==4.8.0.74' \
    'pycocotools>=2.0.7' \
    'scikit-image' \
    'bitsandbytes==0.48.2'

# GCS streaming + huggingface (safe with transformers==4.31.0)
!pip -q install --upgrade gcsfs google-cloud-storage huggingface_hub

# Verify install
import importlib, subprocess
subprocess.run(["python", "-c",
                "import transformers, peft, einops, torch; "
                "print('transformers', transformers.__version__); "
                "print('peft        ', peft.__version__); "
                "print('einops      ', einops.__version__); "
                "print('torch       ', torch.__version__)"])

print("\n>>> RESTART THE RUNTIME NOW <<<")
print("Runtime → Restart session, then run Cells 2 → 8.")


## Cell 2 — authenticate to Google + Hugging Face

* Google auth: `gcsfs` reads images from your private bucket.
* HF login: `checkone/ROSALIA-7B-v1` is ungated but authenticated requests
  get higher rate limits. `xinlai/LISA-7B-v1` (tokenizer) is also ungated.


In [ ]:
from google.colab import auth
auth.authenticate_user()

from huggingface_hub import notebook_login
notebook_login()


## Cell 3 — config

All paths, model IDs, and the PadChest-GR → ROSALIA vocabulary mapping in
one place. ROSALIA supports a fixed 7-lesion vocabulary; findings that
don't map are dropped from the eval set (the paper's own limitation, not
a bug).


In [ ]:
GCS_PROJECT = "yang-uncertainty-eval"
BUCKET      = "yang-padchest-gr"
IMG_PREFIX  = "images"
OUT_PREFIX  = "outputs/rosalia_v1"
MASK_PREFIX = f"{OUT_PREFIX}/masks"

# Drive (for pushing final aggregates back to the repo)
DRIVE_REPO_ROOT = "/content/drive/MyDrive/uncertaintyestimate"

# --- ROSALIA model IDs (all public, ungated on HF) -----------------------
ROSALIA_REPO      = "checkone/ROSALIA-7B-v1"      # main weights
LISA_TOKENIZER    = "xinlai/LISA-7B-v1"           # base tokenizer
CLIP_VISION_TOWER = "openai/clip-vit-large-patch14"

# --- Instruction template ------------------------------------------------
# ROSALIA supports two positive instruction types (§C.1 of the paper):
#   "basic"  : "Segment the [target] in the [location]."
#   "global" : "Segment the [target]."
# We start with "global" because (a) it doesn't require parsing PadChest-GR
# sentences for laterality/zone, and (b) global masks cover ALL instances
# of the finding, which matches how we scored MedSAM-3 (union of ALL GT
# boxes for that sample). If you want tighter localization, switch to
# "basic" and populate a location parser.
INSTRUCTION_MODE       = "global"                 # "global" | "basic"
INSTRUCTION_TEMPLATE_G = "Segment the {target}."
INSTRUCTION_TEMPLATE_B = "Segment the {target} in the {location}."

# --- ROSALIA's 7-lesion vocabulary ---------------------------------------
ROSALIA_LESIONS = {
    "cardiomegaly", "pneumonia", "atelectasis",
    "opacity", "consolidation", "edema", "effusion",
}

# --- PadChest-GR finding_label → ROSALIA target --------------------------
# Conservative mapping: only labels with clear semantic match. Findings
# not present in this dict are DROPPED from the eval set (~30-60% of
# PadChest-GR — mostly anatomical variants like "aortic elongation",
# fibrosis, nodules, granulomas that fall outside ROSALIA's 7-lesion
# vocab). Add more mappings here if you want to expand coverage; each
# addition should map to the ROSALIA concept whose training distribution
# best covers it.
FINDING_TO_ROSALIA = {
    # --- cardiomegaly ---
    "cardiomegaly":                       "cardiomegaly",
    "heart insufficiency":                "cardiomegaly",
    "left ventricular hypertrophy":       "cardiomegaly",

    # --- pneumonia (specific infection pattern) ---
    "pneumonia":                          "pneumonia",
    "atypical pneumonia":                 "pneumonia",

    # --- atelectasis / lung collapse ---
    "atelectasis":                        "atelectasis",
    "laminar atelectasis":                "atelectasis",
    "lobar atelectasis":                  "atelectasis",
    "total atelectasis":                  "atelectasis",
    "segmental atelectasis":              "atelectasis",
    "round atelectasis":                  "atelectasis",

    # --- consolidation ---
    "consolidation":                      "consolidation",
    "pulmonary consolidation":            "consolidation",

    # --- edema ---
    "pulmonary edema":                    "edema",
    "edema":                              "edema",
    # Vascular signs that clinically indicate edema
    "vascular redistribution":            "edema",
    "increased density":                  "edema",

    # --- effusion ---
    "pleural effusion":                   "effusion",
    "loculated pleural effusion":         "effusion",
    "loculated fissural effusion":        "effusion",

    # --- opacity (catch-all, per ROSALIA paper §C.1: opacity is the
    #     high-level supertype for pneumonia/atelectasis/edema when
    #     the specific diagnosis is uncertain) ---
    "opacity":                            "opacity",
    "lung opacity":                       "opacity",
    "alveolar pattern":                   "opacity",
    "interstitial pattern":               "opacity",
    "reticular interstitial pattern":     "opacity",
    "reticulonodular interstitial pattern": "opacity",
    "miliary opacities":                  "opacity",
    "ground glass pattern":               "opacity",
    "air bronchogram":                    "opacity",
    "infiltrates":                        "opacity",
    "infiltrate":                         "opacity",
}

MASK_GRID   = 1024
MASK_THRESHOLD = 0.5

SMOKE_N = 50
SMOKE_SEED = 42

SAMPLES_CSV = "/content/samples_with_uncertainty_and_iou.csv"
RULE_JSONL  = "/content/rule_uncertainty_scores.jsonl"

print(f"Config loaded. Instruction mode: {INSTRUCTION_MODE}")
print(f"ROSALIA vocab: {sorted(ROSALIA_LESIONS)}")
print(f"PadChest-GR → ROSALIA mappings defined: {len(FINDING_TO_ROSALIA)}")


## Cell 4 — load samples, merge rule labels, and filter to ROSALIA vocab

1. Fetch the label files directly from GitHub (~3 MB).
2. Mount Drive (best-effort, only for Cell 12 push-back).
3. Load `samples_with_uncertainty_and_iou.csv` + `rule_uncertainty_scores.jsonl`.
4. Add a `rosalia_target` column via `FINDING_TO_ROSALIA` mapping.
5. Drop rows with no mapping — those findings aren't in ROSALIA's vocab.
6. Report what fraction survived + finding-type breakdown.

There is NO MedGemma step in this notebook. ROSALIA's 7-word vocabulary
is small enough that a deterministic dict lookup is more reliable than
an LLM rewrite (which is also what the ROSALIA paper does — see §3.1).


In [ ]:
REPO_RAW_URL = "https://raw.githubusercontent.com/nprakash1/uncertaintyestimateyang/rule-bag-of-words"
SAMPLES_CSV  = "/content/samples_with_uncertainty_and_iou.csv"
RULE_JSONL   = "/content/rule_uncertainty_scores.jsonl"

import os
if not os.path.exists(SAMPLES_CSV):
    get_ipython().system(
        f"curl -fsSL '{REPO_RAW_URL}/project/data/processed/samples_with_uncertainty_and_iou.csv' "
        f"-o '{SAMPLES_CSV}'"
    )
if not os.path.exists(RULE_JSONL):
    get_ipython().system(
        f"curl -fsSL '{REPO_RAW_URL}/project/data/processed/rule_uncertainty_scores.jsonl' "
        f"-o '{RULE_JSONL}'"
    )
print(f"samples CSV: {os.path.getsize(SAMPLES_CSV):,} bytes")
print(f"rule  JSONL: {os.path.getsize(RULE_JSONL):,} bytes")

# Drive (best-effort, for Cell 12)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
except Exception as e:
    print(f"[warn] Drive not mounted ({e}); Cell 12 will fall back to /content/.")
    DRIVE_MOUNTED = False

import json, pandas as pd
samples = pd.read_csv(SAMPLES_CSV)
print(f"Loaded {len(samples)} rows from samples CSV")
print("Columns:", list(samples.columns))

rule_rows = []
with open(RULE_JSONL, "r") as f:
    for line in f:
        line = line.strip()
        if not line: continue
        try:
            rule_rows.append(json.loads(line))
        except Exception:
            continue
rule_df = pd.DataFrame(rule_rows)[["sample_id", "uncertainty_label"]]
rule_df = rule_df.rename(columns={"uncertainty_label": "uncertainty_label_rule"})
print(f"Loaded {len(rule_df)} rule-based labels")

samples = samples.rename(columns={"uncertainty_label": "uncertainty_label_medgemma"})
samples = samples.merge(rule_df, on="sample_id", how="left")

# --- ROSALIA target mapping ---------------------------------------------
def _map_to_rosalia(label):
    if not isinstance(label, str): return None
    return FINDING_TO_ROSALIA.get(label.strip().lower())

samples["rosalia_target"] = samples["finding_label"].apply(_map_to_rosalia)

n_total   = len(samples)
n_mapped  = samples["rosalia_target"].notna().sum()
n_dropped = n_total - n_mapped
print(f"\nMapping to ROSALIA vocab:")
print(f"  total samples          : {n_total}")
print(f"  mapped to ROSALIA vocab: {n_mapped}  ({100*n_mapped/n_total:.1f}%)")
print(f"  dropped (out of vocab) : {n_dropped}")

# Show top dropped finding_labels so you can decide if you want to extend
# FINDING_TO_ROSALIA in Cell 3
dropped_top = (samples[samples["rosalia_target"].isna()]
               .finding_label.value_counts().head(15))
print("\nTop 15 dropped finding_labels (extend FINDING_TO_ROSALIA to include):")
print(dropped_top.to_string())

samples = samples[samples["rosalia_target"].notna()].reset_index(drop=True)
print(f"\nWorking set after filter: {len(samples)} rows")

# Distribution by ROSALIA target × uncertainty label
print("\nROSALIA target × rule-based uncertainty (stratification cells):")
print(pd.crosstab(samples["rosalia_target"], samples["uncertainty_label_rule"], margins=True))

# --- Build the instruction string ---------------------------------------
def _build_instruction(target, mode=INSTRUCTION_MODE, location=None):
    if mode == "global":
        return INSTRUCTION_TEMPLATE_G.format(target=target)
    elif mode == "basic":
        loc = location if location else "right lung"  # fallback
        return INSTRUCTION_TEMPLATE_B.format(target=target, location=loc)
    else:
        raise ValueError(f"Unknown INSTRUCTION_MODE: {mode}")

samples["rosalia_instruction"] = samples["rosalia_target"].apply(_build_instruction)

samples.head(5)[["sample_id","image_id","finding_label",
                 "rosalia_target","rosalia_instruction",
                 "uncertainty_label_rule"]]


## Cell 5 — GCS streaming image loader

Same as the previous MedSAM-3 notebook. Streams a single PadChest-GR PNG
from `gs://${BUCKET}/images/`, applies percentile-based windowing to
convert 16-bit CXR pixels to 8-bit RGB (16-bit PIL `.convert("RGB")` is
lossy — it clips to [0, 255] and mostly saturates to white for typical
CXR ranges), and returns a `PIL.Image.Image`.

ROSALIA's inference example uses `cv2.imread(...)` (BGR uint8), so in
Cell 8 we convert PIL → `np.array(img)` (RGB uint8) before passing to
the ROSALIA processor.


In [ ]:
import io
import numpy as np
from PIL import Image
import gcsfs

fs = gcsfs.GCSFileSystem(project=GCS_PROJECT)


def _window_to_uint8(arr: np.ndarray, low_pct: float = 1.0, high_pct: float = 99.0) -> np.ndarray:
    arr = arr.astype(np.float32)
    lo, hi = np.percentile(arr, [low_pct, high_pct])
    if hi - lo < 1.0:
        hi = lo + 1.0
    arr = np.clip((arr - lo) / (hi - lo), 0.0, 1.0)
    return (arr * 255.0).astype(np.uint8)


def load_image_gcs(image_id: str) -> Image.Image:
    path = f"{BUCKET}/{IMG_PREFIX}/{image_id}"
    with fs.open(path, "rb") as f:
        data = f.read()
    img = Image.open(io.BytesIO(data))
    arr = np.array(img)
    if arr.ndim == 3:
        arr = arr[..., 0]
    arr = _window_to_uint8(arr)
    return Image.fromarray(arr).convert("RGB")


_test_id = samples.iloc[0]["image_id"]
_img = load_image_gcs(_test_id)
_a = np.array(_img)
print(f"OK — sample image {_test_id}: size={_img.size}, "
      f"uint8 min={_a.min()}, max={_a.max()}, mean={_a.mean():.1f}")
if _a.mean() > 220 or _a.mean() < 30:
    print("[warn] mean pixel intensity is extreme — check windowing")


## Cell 6 — load ROSALIA (LISA-7B + SAM-H fine-tuned on MIMIC-ILS)

We use the custom `LISAForCausalLM` class shipped in the ROSALIA repo,
which extends the LISA-7B architecture (`xinlai/LISA-7B-v1`). The public
HF weights repo is `checkone/ROSALIA-7B-v1` — sharded pytorch_model
(9.98 GB + 6.14 GB).

Pipeline:
1. `sys.path.insert(0, "/content/rosalia_repo")` so we can import
   `model.LISA`, `model.llava.*`, `model.segment_anything.*`.
2. Load the base LISA-7B tokenizer + find `[SEG]` token id.
3. Instantiate `LISAForCausalLM.from_pretrained("checkone/ROSALIA-7B-v1")`
   with `vision_tower="openai/clip-vit-large-patch14"`.
4. Initialize the vision tower + CLIP processor + SAM `ResizeLongestSide`
   transform.

`segment_image_rosalia(model, tokenizer, clip_processor, transform,
pil_image, instruction) -> (mask_np, text_output)`
* `mask_np`: uint8 (H, W) at ORIGINAL image resolution, values in {0, 1}
  (thresholded from `pred_masks[0] > 0`).
* `text_output`: the model's textual reply (e.g. "Sure, [SEG]." or
  "There is no pneumonia in the right lung.").

No query-selection or area-filter heuristics needed — LISA outputs a
single mask per `[SEG]` token, which is exactly what we want.


In [ ]:
import sys, os
sys.path.insert(0, "/content/rosalia_repo")

# Silence the LISA custom logging spam during .evaluate()
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import cv2, numpy as np, torch
import torch.nn.functional as F
import transformers
from transformers import CLIPImageProcessor

# ROSALIA repo imports (require sys.path.insert above)
from model.LISA import LISAForCausalLM
from model.llava import conversation as conversation_lib
from model.llava.mm_utils import tokenizer_image_token
from model.segment_anything.utils.transforms import ResizeLongestSide
from utils.utils import (
    DEFAULT_IM_END_TOKEN,
    DEFAULT_IM_START_TOKEN,
    DEFAULT_IMAGE_TOKEN,
    IMAGE_TOKEN_INDEX,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cpu":
    raise RuntimeError("ROSALIA requires a CUDA GPU. Change runtime to A100/L4.")

IMAGE_SIZE = 1024


def _sam_preprocess(x, img_size=IMAGE_SIZE):
    # Standard SAM preprocess: ImageNet-normalize + pad to square.
    # From inference_rosalia_example.py.
    pixel_mean = torch.tensor([123.675, 116.28, 103.53]).view(-1, 1, 1)
    pixel_std  = torch.tensor([58.395, 57.12, 57.375]).view(-1, 1, 1)
    x = (x - pixel_mean) / pixel_std
    h, w = x.shape[-2:]
    padh = img_size - h
    padw = img_size - w
    x = F.pad(x, (0, padw, 0, padh))
    return x


def load_rosalia():
    # Tokenizer from base LISA-7B (must match — ROSALIA fine-tune reuses it)
    tokenizer = transformers.AutoTokenizer.from_pretrained(
        LISA_TOKENIZER,
        cache_dir=None,
        model_max_length=512,
        padding_side="right",
        use_fast=False,
    )
    tokenizer.pad_token = tokenizer.unk_token
    seg_token_idx = tokenizer("[SEG]", add_special_tokens=False).input_ids[0]

    # Model: LISAForCausalLM from ROSALIA HF repo
    dtype = torch.bfloat16
    model = LISAForCausalLM.from_pretrained(
        ROSALIA_REPO,
        low_cpu_mem_usage=True,
        vision_tower=CLIP_VISION_TOWER,
        seg_token_idx=seg_token_idx,
        torch_dtype=dtype,
    )
    model.config.eos_token_id = tokenizer.eos_token_id
    model.config.bos_token_id = tokenizer.bos_token_id
    model.config.pad_token_id = tokenizer.pad_token_id

    # Initialize vision tower on GPU
    model.get_model().initialize_vision_modules(model.get_model().config)
    vision_tower = model.get_model().get_vision_tower()
    vision_tower.to(dtype=dtype, device=0)

    model = model.bfloat16().cuda()
    model.eval()

    clip_processor = CLIPImageProcessor.from_pretrained(model.config.vision_tower)
    transform      = ResizeLongestSide(IMAGE_SIZE)

    print("ROSALIA loaded. VRAM after load:")
    if torch.cuda.is_available():
        print(f"  allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        print(f"  reserved : {torch.cuda.memory_reserved()/1e9:.2f} GB")
    return model, tokenizer, clip_processor, transform


model, tokenizer, clip_processor, transform = load_rosalia()


def segment_image_rosalia(model, tokenizer, clip_processor, transform,
                          pil_image, instruction: str):
    # PIL -> RGB uint8 numpy (same shape ROSALIA expects from cv2.imread + BGR2RGB)
    image_np = np.array(pil_image)
    if image_np.ndim == 2:
        image_np = np.stack([image_np, image_np, image_np], axis=-1)
    elif image_np.shape[-1] == 4:
        image_np = image_np[..., :3]
    original_size_list = [image_np.shape[:2]]

    # Build the conversation prompt using LISA's llava_v1 template
    conv = conversation_lib.conv_templates["llava_v1"].copy()
    conv.messages = []
    prompt = DEFAULT_IMAGE_TOKEN + "\n" + instruction
    replace_token = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN
    prompt = prompt.replace(DEFAULT_IMAGE_TOKEN, replace_token)
    conv.append_message(conv.roles[0], prompt)
    conv.append_message(conv.roles[1], "")
    prompt = conv.get_prompt()

    # CLIP image tensor
    image_clip = (
        clip_processor.preprocess(image_np, return_tensors="pt")["pixel_values"][0]
        .unsqueeze(0).cuda().bfloat16()
    )

    # SAM image tensor (resize longest edge to 1024, then normalize+pad)
    image_resized = transform.apply_image(image_np)
    resize_list = [image_resized.shape[:2]]
    image = (
        _sam_preprocess(torch.from_numpy(image_resized).permute(2, 0, 1).contiguous())
        .unsqueeze(0).cuda().bfloat16()
    )

    # Tokenize (handles <image> insertion via IMAGE_TOKEN_INDEX)
    input_ids = tokenizer_image_token(prompt, tokenizer, return_tensors="pt")
    input_ids = input_ids.unsqueeze(0).cuda()

    with torch.no_grad():
        output_ids, pred_masks = model.evaluate(
            image_clip, image, input_ids,
            resize_list, original_size_list,
            max_new_tokens=256,
            tokenizer=tokenizer,
        )

    # Decode the model's textual answer
    out_ids = output_ids[0][output_ids[0] != IMAGE_TOKEN_INDEX]
    text_out = tokenizer.decode(out_ids, skip_special_tokens=False)
    text_out = text_out.replace("\n", "").replace("  ", " ")
    text_out = text_out.split("ASSISTANT:")[-1].split("</s>")[0].strip()

    # Mask: pred_masks is a list of tensors; index 0 is the mask for the
    # first (and only) [SEG] token. Shape is (H_orig, W_orig).
    if len(pred_masks) == 0 or pred_masks[0].numel() == 0:
        mask = np.zeros(original_size_list[0], dtype=np.uint8)
    else:
        mask = (pred_masks[0] > 0).detach().cpu().numpy().astype(np.uint8)
        # Some LISA variants return (1, H, W); squeeze if needed
        if mask.ndim == 3:
            mask = mask[0]

    return mask, text_out


# Quick smoke inference on the first sample so you catch OOM / model-load
# bugs before Cell 8 loops over 50 images.
_row = samples.iloc[0]
_img = load_image_gcs(_row.image_id)
print(f"\nSmoke: image_id={_row.image_id}, instruction={_row.rosalia_instruction!r}")
_mask, _text = segment_image_rosalia(model, tokenizer, clip_processor, transform,
                                     _img, _row.rosalia_instruction)
print(f"  mask shape={_mask.shape}, sum={int(_mask.sum())} pixels")
print(f"  text_output: {_text!r}")


## Cell 7 — IoU helpers

Identical to the MedSAM-3 notebook — mirrors `project/src/compute_iou.py`
so the notebook is fully self-contained.


In [ ]:
import numpy as np
from itertools import combinations
from typing import List

def boxes_to_mask(boxes, grid_size=MASK_GRID):
    mask = np.zeros((grid_size, grid_size), dtype=np.uint8)
    if not boxes: return mask
    for box in boxes:
        x1, y1, x2, y2 = box
        px1 = max(0, min(grid_size, int(np.floor(x1*grid_size))))
        py1 = max(0, min(grid_size, int(np.floor(y1*grid_size))))
        px2 = max(0, min(grid_size, int(np.ceil (x2*grid_size))))
        py2 = max(0, min(grid_size, int(np.ceil (y2*grid_size))))
        if px2 > px1 and py2 > py1:
            mask[py1:py2, px1:px2] = 1
    return mask

def mask_iou(m1, m2):
    inter = int(np.logical_and(m1, m2).sum())
    union = int(np.logical_or (m1, m2).sum())
    return inter/union if union > 0 else float("nan")

def boxes_to_intersection_mask(boxes, grid_size=MASK_GRID):
    if not boxes: return np.zeros((grid_size, grid_size), np.uint8)
    masks = [boxes_to_mask([b], grid_size) for b in boxes]
    out = masks[0].copy()
    for m in masks[1:]:
        out = np.logical_and(out, m).astype(np.uint8)
    return out

def boxes_to_outer_bbox_mask(boxes, grid_size=MASK_GRID):
    if not boxes: return np.zeros((grid_size, grid_size), np.uint8)
    xs1 = [b[0] for b in boxes]; ys1 = [b[1] for b in boxes]
    xs2 = [b[2] for b in boxes]; ys2 = [b[3] for b in boxes]
    return boxes_to_mask([[min(xs1), min(ys1), max(xs2), max(ys2)]], grid_size)

def pairwise_box_ious(boxes, grid_size=MASK_GRID):
    if not boxes or len(boxes) < 2: return []
    masks = [boxes_to_mask([b], grid_size) for b in boxes]
    return [mask_iou(masks[i], masks[j]) for i,j in combinations(range(len(masks)), 2)]

def pred_probs_to_grid_mask(probs, grid_size=MASK_GRID, threshold=MASK_THRESHOLD):
    arr = np.asarray(probs)
    if arr.ndim == 4 and arr.shape[0] == 1: arr = arr[0]
    if arr.ndim == 3: arr = arr.max(axis=0)
    if arr.ndim != 2: raise ValueError(f"Bad shape {probs.shape}")
    try:
        from skimage.transform import resize
        resized = resize(arr.astype(np.float32), (grid_size, grid_size),
                         order=1, mode="edge", anti_aliasing=False,
                         preserve_range=True)
    except Exception:
        h, w = arr.shape
        ys = np.linspace(0, h-1, grid_size).astype(np.int64)
        xs = np.linspace(0, w-1, grid_size).astype(np.int64)
        resized = arr[ys[:, None], xs[None, :]]
    return (resized > threshold).astype(np.uint8)

def compute_iou_bundle(pred_probs, r1_boxes, r2_boxes,
                       grid_size=MASK_GRID, threshold=MASK_THRESHOLD):
    pm = pred_probs_to_grid_mask(pred_probs, grid_size, threshold)
    all_boxes = (r1_boxes or []) + (r2_boxes or [])
    per_box = [mask_iou(pm, boxes_to_mask([b], grid_size)) for b in all_boxes]
    return {
        "per_box_ious":            per_box,
        "max_per_box_iou":         float(np.max(per_box))  if per_box else float("nan"),
        "mean_per_box_iou":        float(np.mean(per_box)) if per_box else float("nan"),
        "iou_with_pixel_or_union": mask_iou(pm, boxes_to_mask(all_boxes, grid_size)),
        "iou_with_outer_bbox_union": mask_iou(pm, boxes_to_outer_bbox_mask(all_boxes, grid_size)),
        "iou_with_intersection":   mask_iou(pm, boxes_to_intersection_mask(all_boxes, grid_size)),
        "iou_with_reader1_union":  mask_iou(pm, boxes_to_mask(r1_boxes or [], grid_size)),
        "iou_with_reader2_union":  mask_iou(pm, boxes_to_mask(r2_boxes or [], grid_size)),
        "reader_iou_union":        mask_iou(boxes_to_mask(r1_boxes or [], grid_size),
                                            boxes_to_mask(r2_boxes or [], grid_size)),
        "pairwise_box_ious":       pairwise_box_ious(all_boxes, grid_size),
        "mean_pairwise_iou":       float(np.mean(pairwise_box_ious(all_boxes, grid_size)))
                                   if len(all_boxes) >= 2 else float("nan"),
        "pred_area":               int(pm.sum()),
        "num_boxes_total":         len(all_boxes),
        "num_reader1_boxes":       len(r1_boxes or []),
        "num_reader2_boxes":       len(r2_boxes or []),
    }, pm


## Cell 8 — SMOKE TEST (50 samples, ~2–4 min on A100)

Stratified sample of 25 uncertain + 25 certain from the filtered
(ROSALIA-vocab) working set. Runs end-to-end + prints the first 5 IoU
rows + saves a 5×4 figure so you can eyeball whether ROSALIA is
producing sensible lesion-shaped masks.

If ROSALIA replies "There is no [target]." for a sample, `pred_masks[0]`
will be all zeros and IoU vs any non-empty GT will be 0. Look at the
`text_output` column to distinguish "model said no" from "model failed".


In [ ]:
import random, json
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as patches

random.seed(SMOKE_SEED)


def stratified_sample(df, n_per_class):
    out = []
    for cls in ["certain", "uncertain"]:
        sub = df[df["uncertainty_label_rule"] == cls]
        out.append(sub.sample(n=min(n_per_class, len(sub)), random_state=SMOKE_SEED))
    return pd.concat(out).sample(frac=1, random_state=SMOKE_SEED).reset_index(drop=True)

smoke = stratified_sample(samples, SMOKE_N // 2)
print(f"Smoke set: {len(smoke)} samples "
      f"({(smoke.uncertainty_label_rule=='uncertain').sum()} uncertain, "
      f"{(smoke.uncertainty_label_rule=='certain').sum()} certain)")

smoke_rows  = []
sample_masks = {}
sample_probs = {}

for row in tqdm(smoke.itertuples(), total=len(smoke), desc="smoke inference"):
    img = load_image_gcs(row.image_id)
    mask, text_out = segment_image_rosalia(
        model, tokenizer, clip_processor, transform,
        img, row.rosalia_instruction,
    )
    r1 = json.loads(row.reader1_boxes) if isinstance(row.reader1_boxes, str) else row.reader1_boxes
    r2 = json.loads(row.reader2_boxes) if isinstance(row.reader2_boxes, str) else row.reader2_boxes
    bundle, pm = compute_iou_bundle(mask, r1, r2)
    rec = {
        "sample_id": row.sample_id,
        "image_id":  row.image_id,
        "finding_label": row.finding_label,
        "rosalia_target": row.rosalia_target,
        "rosalia_instruction": row.rosalia_instruction,
        "rosalia_text_output": text_out,
        "uncertainty_label_rule":     row.uncertainty_label_rule,
        "uncertainty_label_medgemma": row.uncertainty_label_medgemma,
        "sentence_raw":   row.sentence,
        **bundle,
    }
    smoke_rows.append(rec)
    sample_masks[row.sample_id] = pm
    sample_probs[row.sample_id] = mask

smoke_df = pd.DataFrame(smoke_rows)
print("\nFirst 5 IoU rows:")
display(smoke_df[[
    "sample_id","uncertainty_label_rule","rosalia_target",
    "rosalia_text_output",
    "iou_with_pixel_or_union","iou_with_outer_bbox_union",
    "mean_per_box_iou","reader_iou_union","num_boxes_total",
]].head(5))

# --- 5×4 sanity-check figure ---------------------------------------------
fig, axes = plt.subplots(5, 4, figsize=(16, 20))
for i, row in enumerate(smoke_df.head(5).itertuples()):
    img = load_image_gcs(row.image_id)
    iw, ih = img.size
    r1 = json.loads(samples.loc[samples.sample_id==row.sample_id, "reader1_boxes"].iloc[0])
    r2 = json.loads(samples.loc[samples.sample_id==row.sample_id, "reader2_boxes"].iloc[0])
    pm = sample_masks[row.sample_id]

    axes[i,0].imshow(img, cmap="gray"); axes[i,0].set_title("image"); axes[i,0].axis("off")

    axes[i,1].imshow(img, cmap="gray")
    for b, c in [(r1, "red"), (r2, "blue")]:
        for x1,y1,x2,y2 in b:
            axes[i,1].add_patch(patches.Rectangle(
                (x1*iw, y1*ih), (x2-x1)*iw, (y2-y1)*ih,
                lw=2, edgecolor=c, facecolor="none"))
    axes[i,1].set_title("GT boxes (R1=red, R2=blue)"); axes[i,1].axis("off")

    axes[i,2].imshow(pm, cmap="gray"); axes[i,2].set_title(
        f"pred mask  IoU(or)={row.iou_with_pixel_or_union:.2f}"); axes[i,2].axis("off")

    axes[i,3].imshow(img, cmap="gray")
    pm_disp = np.array(Image.fromarray((pm*255).astype(np.uint8)).resize((iw, ih), Image.NEAREST))
    axes[i,3].imshow(np.ma.masked_where(pm_disp == 0, pm_disp),
                     cmap="Reds", alpha=0.5)
    axes[i,3].set_title(f"[{row.uncertainty_label_rule}] target={row.rosalia_target}\n"
                        f"{row.rosalia_text_output[:60]}")
    axes[i,3].axis("off")

plt.tight_layout()
plt.savefig("/content/rosalia_smoke_grid.png", dpi=110)
plt.show()
print("Saved smoke figure: /content/rosalia_smoke_grid.png")


## Cell 9 — FULL RUN (resumable)

Same pattern as before: for each row in the ROSALIA-vocab-filtered
`samples`, check if the per-sample JSON already exists in GCS
(`outputs/rosalia_v1/{sample_id}.json`) and skip if so. Otherwise stream
the image, run ROSALIA, compute the IoU bundle, upload the JSON + a
`.npz` of the predicted mask.

Expected wall-clock: ~2 s per sample on A100 → 30-60 min for the
filtered PadChest-GR subset (depends on how many finding_labels you
mapped in Cell 3).


In [ ]:
import io, json
import numpy as np
from tqdm.auto import tqdm


def exists_in_bucket(path):
    try:
        return fs.exists(path)
    except Exception:
        return False

def upload_json(path, obj):
    with fs.open(path, "w") as f:
        f.write(json.dumps(obj))

def upload_npz(path, mask):
    buf = io.BytesIO()
    np.savez_compressed(buf, mask=mask)
    buf.seek(0)
    with fs.open(path, "wb") as f:
        f.write(buf.read())

all_records = []
n_skipped = 0
n_new = 0

for row in tqdm(samples.itertuples(), total=len(samples), desc="full inference"):
    out_json = f"{BUCKET}/{OUT_PREFIX}/{row.sample_id}.json"
    out_npz  = f"{BUCKET}/{MASK_PREFIX}/{row.sample_id}.npz"
    if exists_in_bucket(out_json):
        with fs.open(out_json, "r") as f:
            all_records.append(json.loads(f.read()))
        n_skipped += 1
        continue
    try:
        img = load_image_gcs(row.image_id)
        mask, text_out = segment_image_rosalia(
            model, tokenizer, clip_processor, transform,
            img, row.rosalia_instruction,
        )
        r1 = json.loads(row.reader1_boxes) if isinstance(row.reader1_boxes, str) else row.reader1_boxes
        r2 = json.loads(row.reader2_boxes) if isinstance(row.reader2_boxes, str) else row.reader2_boxes
        bundle, pm = compute_iou_bundle(mask, r1, r2)
        rec = {
            "sample_id": row.sample_id,
            "image_id":  row.image_id,
            "finding_label": row.finding_label,
            "rosalia_target": row.rosalia_target,
            "rosalia_instruction": row.rosalia_instruction,
            "rosalia_text_output": text_out,
            "uncertainty_label_rule":     row.uncertainty_label_rule,
            "uncertainty_label_medgemma": row.uncertainty_label_medgemma,
            "sentence_raw":   row.sentence,
            **bundle,
        }
        upload_json(out_json, rec)
        upload_npz (out_npz,  pm)
        all_records.append(rec)
        n_new += 1
    except Exception as e:
        print(f"[skip] {row.sample_id}: {type(e).__name__}: {e}")
        continue

results_df = pd.DataFrame(all_records)
print(f"\nDone. Cached/skipped: {n_skipped}.  New this session: {n_new}.  "
      f"Total in results_df: {len(results_df)}.")
results_df.to_csv("/content/rosalia_per_sample_ious.csv", index=False)
print("Wrote /content/rosalia_per_sample_ious.csv")


## Cell 10 — aggregates stratified by rule-based uncertainty

Same shape as before — group-level table + per-finding × group + bootstrapped
95% CI for `iou_with_pixel_or_union` mean per group. Additionally reports the
per-`rosalia_target` breakdown since ROSALIA's per-lesion performance varies
substantially (Table 5 in the paper: cardiomegaly 0.89 vs pneumonia 0.57).


In [ ]:
import json, numpy as np
RNG = np.random.default_rng(42)
BOOT_N = 2000

IOU_COLS = [
    "iou_with_pixel_or_union",
    "iou_with_outer_bbox_union",
    "iou_with_intersection",
    "iou_with_reader1_union",
    "iou_with_reader2_union",
    "reader_iou_union",
    "mean_per_box_iou",
    "max_per_box_iou",
    "mean_pairwise_iou",
]

for c in IOU_COLS:
    results_df[c] = pd.to_numeric(results_df[c], errors="coerce")

# --- group-level table ----------------------------------------------------
group_agg = results_df.groupby("uncertainty_label_rule")[IOU_COLS].agg(
    ["count","mean","median","std"]
)
print("=== By uncertainty group ===")
print(group_agg)

# --- per-ROSALIA-target × group ------------------------------------------
print("\n=== By ROSALIA target × uncertainty group ===")
pt = results_df.groupby(
    ["rosalia_target","uncertainty_label_rule"]
)[IOU_COLS].agg(["count","mean","median"])
print(pt)
pt.to_csv("/content/rosalia_per_target_ious.csv")

# --- per-finding × group table --------------------------------------------
pf = results_df.groupby(
    ["uncertainty_label_rule","finding_label"]
)[IOU_COLS].agg(["count","mean","median"])
pf.to_csv("/content/rosalia_per_finding_ious.csv")

# --- bootstrapped CI on iou_with_pixel_or_union mean ----------------------
def boot_mean_ci(x, n=BOOT_N, alpha=0.05):
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]
    if len(x) == 0: return (float("nan"),)*3
    idx = RNG.integers(0, len(x), size=(n, len(x)))
    means = x[idx].mean(axis=1)
    lo, hi = np.quantile(means, [alpha/2, 1-alpha/2])
    return float(means.mean()), float(lo), float(hi)

agg_json = {}
for cls, sub in results_df.groupby("uncertainty_label_rule"):
    mean, lo, hi = boot_mean_ci(sub["iou_with_pixel_or_union"])
    agg_json[str(cls)] = {
        "n": int(len(sub)),
        "iou_with_pixel_or_union_mean": mean,
        "iou_with_pixel_or_union_ci95": [lo, hi],
        "iou_with_outer_bbox_union_mean": float(sub["iou_with_outer_bbox_union"].mean()),
        "iou_with_intersection_mean":     float(sub["iou_with_intersection"].mean()),
        "reader_iou_union_mean":          float(sub["reader_iou_union"].mean()),
        "mean_per_box_iou_mean":          float(sub["mean_per_box_iou"].mean()),
    }
with open("/content/rosalia_aggregate_ious.json", "w") as f:
    json.dump(agg_json, f, indent=2)
agg_json


## Cell 11 — plots

Three figures:
1. Histogram of `iou_with_pixel_or_union` by uncertainty group.
2. Per-group mean IoU bar chart across metrics.
3. Scatter: annotator agreement (`reader_iou_union`) vs ROSALIA IoU,
   colored by group.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7,4))
for cls, color in [("certain","tab:blue"), ("uncertain","tab:orange")]:
    sub = results_df[results_df["uncertainty_label_rule"]==cls]["iou_with_pixel_or_union"].dropna()
    plt.hist(sub, bins=40, alpha=0.55, label=f"{cls} (n={len(sub)})", color=color)
plt.xlabel("IoU(pred, pixel-OR union of GT boxes)")
plt.ylabel("Count")
plt.title("ROSALIA IoU vs PadChest-GR, stratified by rule-based spatial uncertainty")
plt.legend()
plt.tight_layout()
plt.savefig("/content/rosalia_iou_histogram_by_group.png", dpi=120)
plt.show()

plot_cols = [
    "iou_with_pixel_or_union",
    "iou_with_outer_bbox_union",
    "iou_with_intersection",
    "mean_per_box_iou",
    "max_per_box_iou",
    "reader_iou_union",
]
means = results_df.groupby("uncertainty_label_rule")[plot_cols].mean()
fig, ax = plt.subplots(figsize=(10,4))
means.T.plot(kind="bar", ax=ax)
ax.set_ylabel("Mean IoU")
ax.set_title("ROSALIA — Mean IoU per metric, by rule-based uncertainty group")
ax.set_xticklabels(plot_cols, rotation=30, ha="right")
plt.tight_layout()
plt.savefig("/content/rosalia_iou_by_uncertainty_group.png", dpi=120)
plt.show()

plt.figure(figsize=(6,5))
for cls, color in [("certain","tab:blue"), ("uncertain","tab:orange")]:
    sub = results_df[results_df["uncertainty_label_rule"]==cls]
    plt.scatter(sub["reader_iou_union"], sub["iou_with_pixel_or_union"],
                s=10, alpha=0.5, label=cls, color=color)
plt.xlabel("Reader-Reader IoU (annotator agreement)")
plt.ylabel("ROSALIA IoU (pred vs union of GT boxes)")
plt.title("ROSALIA quality vs annotator agreement")
plt.legend()
plt.tight_layout()
plt.savefig("/content/rosalia_agreement_vs_iou_scatter.png", dpi=120)
plt.show()


## Cell 12 — push aggregated results back to the repo via Drive

Copies the final artifacts to Drive so they appear under `project/outputs/`
and `project/figures/` after a git-side rsync on your laptop.


In [ ]:
import shutil, os

if globals().get("DRIVE_MOUNTED", False) and os.path.isdir("/content/drive/MyDrive"):
    out_dir = f"{DRIVE_REPO_ROOT}/project/outputs"
    fig_dir = f"{DRIVE_REPO_ROOT}/project/figures"
    print(f"Writing to Drive: {DRIVE_REPO_ROOT}")
else:
    out_dir = "/content/outputs"
    fig_dir = "/content/figures"
    print("Drive not mounted — writing to /content/. Use File → Download in Colab.")

os.makedirs(out_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)

for fn in ("rosalia_per_sample_ious.csv",
           "rosalia_per_finding_ious.csv",
           "rosalia_per_target_ious.csv",
           "rosalia_aggregate_ious.json"):
    src = f"/content/{fn}"
    if os.path.exists(src):
        shutil.copy(src, f"{out_dir}/{fn}")

for fn in ("rosalia_iou_histogram_by_group.png",
           "rosalia_iou_by_uncertainty_group.png",
           "rosalia_agreement_vs_iou_scatter.png",
           "rosalia_smoke_grid.png"):
    src = f"/content/{fn}"
    if os.path.exists(src):
        shutil.copy(src, f"{fig_dir}/{fn}")

print(f"Done. outputs → {out_dir}, figures → {fig_dir}")
